# 01 — Molecular identity review

Review molecular identity only after the ROI-to-anatomy geometry has been accepted. This notebook checks that adding identity did not change geometry.

In [ ]:
# 02 — Choose the fish and saved run
from pathlib import Path
from codeants_2pf_hcr.qc_notebooks import QCNotebookConfig, resolve_qc_notebook_context

FISH_ID = "L765_f04"
LOCAL_ROOT = Path("/Volumes/dataDrive/dataProcessing/2p_processing")
PIPELINE_ROOT = Path("/Volumes/dataDrive/dataProcessing/2p_processing/_staged_pipeline_runs/20260803T112035Z_functional_registration_ants_no_fallback/L765_f04")
CONTEXT = resolve_qc_notebook_context(QCNotebookConfig(FISH_ID, LOCAL_ROOT, PIPELINE_ROOT))
GEOMETRY_STATE = "frozen"  # accepted geometry is frozen before identity review


In [ ]:
# 03 — Locate the geometry and identity tables
FROZEN_GEOMETRY_TABLE = CONTEXT.stage_paths["match-roi-to-anatomy"] / "registration" / "functional_roi_anatomy_matches.csv"
IDENTITY_TABLE = CONTEXT.stage_paths["score-activity-bpi"] / "registration" / "functional_roi_activity_identity.csv"
ANATOMY_LABELS = CONTEXT.fish_dir / "03_analysis" / "structural" / "cp_masks" / f"{FISH_ID}_anatomy_00001_uint8_8bit_cp_masks.tif"
HCR_ANATOMY_DIR = CONTEXT.stage_paths["register-hcr-to-anatomy"] / "confocal" / "aligned"
HCR_LABEL_MASKS = sorted(HCR_ANATOMY_DIR.glob("*_cp_masks_in_2p_labels_uint16.tif"))
HCR_FINAL_PAIRS = [path.with_name(path.name.replace("_labels_uint16.tif", "_final_pairs.csv")) for path in HCR_LABEL_MASKS]


In [ ]:
# 04 — Inspect accepted HCR/anatomy centroid offsets by gene
from IPython.display import display
from codeants_2pf_hcr.plots.qc_molecular import (
    build_hcr_anatomy_centroid_offset_table,
    plot_hcr_anatomy_centroid_offsets,
)

HCR_CENTROID_OFFSETS = build_hcr_anatomy_centroid_offset_table(
    anatomy_labels_path=ANATOMY_LABELS,
    hcr_label_paths=HCR_LABEL_MASKS,
    final_pair_paths=HCR_FINAL_PAIRS,
)
HCR_OFFSET_SAMPLE_SIZES = (
    HCR_CENTROID_OFFSETS.loc[HCR_CENTROID_OFFSETS["axis"].eq("x")]
    .groupby("gene").size().rename("n_accepted_pairs").reset_index()
)
display(HCR_OFFSET_SAMPLE_SIZES)
plot_hcr_anatomy_centroid_offsets(HCR_CENTROID_OFFSETS);


The left violin combines X and Y into one lateral XY distance for each already accepted one-to-one HCR/anatomy pair; faint points retain the individual-pair values. Gene labels and the table report the number of accepted pairs. Its dashed orange reference is the median anatomy-mask radius, measured from each label's largest XY cross-section in physical units. Z is shown separately because the confocal point-spread function is broader axially. This is review evidence only and does not exclude an accepted pair.

In [ ]:
# 05 — Compare geometry before and after identity assignment
from IPython.display import display
from codeants_2pf_hcr.plots.qc_molecular import inspect_molecular_identity_qc, plot_molecular_identity_qc

report = inspect_molecular_identity_qc(
    fish_id=FISH_ID,
    pipeline_root=CONTEXT.pipeline_root,
    fish_dir=CONTEXT.fish_dir,
    geometry_state=GEOMETRY_STATE,
    frozen_geometry_table_path=FROZEN_GEOMETRY_TABLE,
    identity_table_path=IDENTITY_TABLE,
)
display(report["issues"], report["identity_summary"], report["geometry_changes"])
plot_molecular_identity_qc(report);


## 06 — Decide whether identity review can proceed

Proceed only when geometry is accepted and `geometry_changes` is empty. Identity may be accepted, rejected, uncertain, coexpressing, or unmatched, but it must not change geometry.

In [ ]:
# 06 — Review persisted molecular-to-functional correspondences
from codeants_2pf_hcr.plots.qc_molecular import plot_molecular_correspondence_tiles

FUNCTIONAL_ANATOMY_LABEL_DIR = CONTEXT.stage_paths["transform-functional-rois-to-anatomy"] / "functional" / "anatomy"
TILE_GENES = None  # e.g. ["sst1.2"] — display filter only
TILE_RESPONSE_STATUSES = None  # e.g. ["responsive"] — display filter only
TILE_FLAGGED_ONLY = False  # keeps all persisted links unless explicitly narrowed for review
TILE_MAX = None  # None renders every selected link; use an explicit integer only for a bounded pass
plot_molecular_correspondence_tiles(anatomy_path=CONTEXT.fish_dir / "02_reg" / "00_preprocessing" / "2p_anatomy" / f"{FISH_ID}_anatomy_2P_GCaMP.nrrd", anatomy_labels_path=ANATOMY_LABELS, hcr_label_paths=HCR_LABEL_MASKS, final_pair_paths=HCR_FINAL_PAIRS, identity_table_path=IDENTITY_TABLE, functional_label_dir=FUNCTIONAL_ANATOMY_LABEL_DIR, fish_id=FISH_ID, genes=TILE_GENES, response_statuses=TILE_RESPONSE_STATUSES, flagged_only=TILE_FLAGGED_ONLY, max_tiles=TILE_MAX);


## 07 — Molecular label fate

This HCR-centric view follows every segmented label through frozen HCR-to-anatomy pairs and ROI-centric activity rows. It is review-only and preserves the complete zero-count category domain.

In [ ]:
# 08 — Inspect why molecular labels are or are not represented in functional analysis
from IPython.display import display
from codeants_2pf_hcr.plots.qc_molecular import build_molecular_identity_fate_table, plot_molecular_identity_fate

MOLECULAR_FATES = build_molecular_identity_fate_table(label_paths=HCR_LABEL_MASKS, final_pair_paths=HCR_FINAL_PAIRS, identity_table_path=IDENTITY_TABLE, geometry_table_path=FROZEN_GEOMETRY_TABLE, anatomy_labels_path=ANATOMY_LABELS)
display(MOLECULAR_FATES)
plot_molecular_identity_fate(MOLECULAR_FATES, fish_id=FISH_ID);
